# NaturalisticDiffInt — 01: NMPH Toy Validation

**Goal:** Reproduce the qualitative behaviour of the Ritvo et al. (2024) model in PyTorch:
- Sweep oscillation amplitude (`osc_amp`) from low → high
- Verify the U-shaped outcome: no change → differentiation → integration
- Reproduce prediction that differentiation yields anticorrelated hidden representations
- Validate BCM vs NMPH piecewise learning rules

This notebook contains no external data dependencies. All code is self-contained.

**Reference:** Ritvo, Nguyen, Turk-Browne & Norman (2024). *A neural network model of differentiation and integration of competing memories.* eLife 12:RP88608. DOI: 10.7554/eLife.88608


## 0. GitHub Sync — Setup
Run once per session. Requires `GITHUB_TOKEN` in Colab Secrets.

In [ ]:
# =============================================================================
#  GitHub Sync — Setup  (run once per session)
# =============================================================================
from google.colab import userdata
import os, subprocess

GITHUB_USER  = "drgzkr"
GITHUB_REPO  = "NaturalisticDiffInt"
REPO_PATH    = f"/content/{GITHUB_REPO}"
NOTEBOOK_REL = "notebooks/exploratory/01_nmph_toy_validation.ipynb"

_token  = userdata.get("GITHUB_TOKEN")
_remote = f"https://{_token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

subprocess.run(["git", "config", "--global", "user.name",  "Colab"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab@naturalistic-diffint.local"], check=True)

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", _remote, REPO_PATH], check=True)
    print(f"Cloned  -> {REPO_PATH}")
else:
    subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", _remote])
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
    print(f"Pulled  -> {REPO_PATH}")

del _token, _remote
print(f"Repo ready at {REPO_PATH}")


## 1. Imports and inline model code

The NMPH model is inlined here so the notebook is self-contained in Colab without a pip install step. The canonical source lives in `src/core/` in the repo.

In [ ]:
import sys, os
sys.path.insert(0, REPO_PATH)          # use repo src/ if available

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dataclasses import dataclass, field
from typing import Optional, Literal

print(f"torch {torch.__version__}")
torch.manual_seed(42)
np.random.seed(42)


### KWTALayer — inhibitory dynamics with oscillatory envelope

In [ ]:
class KWTALayer(nn.Module):
    """
    k-Winners-Take-All with sinusoidal oscillatory inhibition.
    See Ritvo et al. (2024) Table 1 and O'Reilly & Munakata (2000).
    """
    def __init__(self, k, k_max=None, target_diff=0.05, osc_amp=0.0, osc_period=10, gain=100.0):
        super().__init__()
        self.k = k
        self.k_max = k_max if k_max is not None else 2 * k
        self.target_diff = target_diff
        self.osc_amp = osc_amp
        self.osc_period = osc_period
        self.gain = gain

    def oscillation_factor(self, t):
        if self.osc_amp == 0.0:
            return 1.0
        return 1.0 - self.osc_amp * math.sin(2 * math.pi * t / self.osc_period)

    def apply(self, x, t=0):
        n = x.shape[0]
        osc = self.oscillation_factor(t)
        sorted_idx = torch.argsort(x, descending=True)
        k_eff = min(self.k, n)
        threshold_excitation = x[sorted_idx[k_eff - 1]]
        inhib = threshold_excitation.item() * osc
        net = x - inhib
        active_mask = (net > 0)
        n_active = active_mask.sum().item()
        if n_active > self.k_max:
            top_k_max_threshold = x[sorted_idx[self.k_max - 1]]
            inhib = top_k_max_threshold.item() * osc
            net = x - inhib
            active_mask = (net > 0)
        net_clamped = torch.clamp(net, -5.0, 5.0)
        act = torch.sigmoid(self.gain * net_clamped) * active_mask.float()
        return act

    def forward(self, x, t=0):
        return self.apply(x, t)


### BCM and NMPH piecewise learning rules

In [ ]:
class BCMLearningRule(nn.Module):
    """
    BCM rule: δw = η * y * (y − θ_M) * x
    θ_M updated as EMA of y² (sliding modification threshold).
    """
    def __init__(self, lr=0.01, tau=0.9, theta_init=0.5, max_w=1.0, min_w=0.0):
        super().__init__()
        self.lr = lr; self.tau = tau
        self.max_w = max_w; self.min_w = min_w
        self.register_buffer("theta_M", torch.tensor(theta_init))

    def update_threshold(self, post_act):
        self.theta_M = self.tau * self.theta_M + (1 - self.tau) * (post_act**2).mean()

    def delta_w(self, pre, post):
        return self.lr * torch.outer(post * (post - self.theta_M), pre)

    def apply(self, w, pre, post):
        dw = self.delta_w(pre, post)
        self.update_threshold(post)
        return torch.clamp(w + dw, self.min_w, self.max_w)

    def forward(self, w, pre, post):
        return self.apply(w, pre, post)


class NMPHLearningRule(nn.Module):
    """
    Piecewise U-shaped NMPH rule based on coactivity c = act_x * act_y.
      c < theta_low              → no change
      theta_low ≤ c < theta_cross → weakening (differentiation)
      c ≥ theta_cross            → strengthening (integration)
    """
    def __init__(self, theta_low=0.05, theta_cross=0.3, lr_weak=0.05, lr_strong=0.02,
                 max_w=1.0, min_w=0.0):
        super().__init__()
        self.theta_low = theta_low; self.theta_cross = theta_cross
        self.lr_weak = lr_weak; self.lr_strong = lr_strong
        self.max_w = max_w; self.min_w = min_w

    def _u_shape(self, c):
        dw = torch.zeros_like(c)
        weak   = (c >= self.theta_low) & (c < self.theta_cross)
        strong = c >= self.theta_cross
        dw[weak]   = -self.lr_weak   * (c[weak]   - self.theta_low)
        dw[strong] = +self.lr_strong * (c[strong] - self.theta_cross)
        return dw

    def apply(self, w, pre, post):
        coactivity = torch.outer(post, pre)
        return torch.clamp(w + self._u_shape(coactivity), self.min_w, self.max_w)

    def forward(self, w, pre, post):
        return self.apply(w, pre, post)


### NMPHNetwork — full four-layer model

In [ ]:
@dataclass
class NetworkConfig:
    n_category: int = 1
    n_item: int = 2
    n_hidden: int = 14
    n_output: int = 1
    hidden_k: int = 6;    hidden_k_max: int = 10
    hidden_target_diff: float = 0.03
    hidden_osc_amp: float = 0.11;  hidden_osc_period: int = 10
    output_k: int = 6;   output_k_max: int = 15
    output_target_diff: float = 0.05; output_osc_amp: float = 0.115
    learning_rule: str = "bcm"
    lr: float = 0.01;  tau_bcm: float = 0.9
    prewire_strength: float = 0.99; hidden_overlap: int = 2


class NMPHNetwork(nn.Module):
    def __init__(self, cfg=None):
        super().__init__()
        self.cfg = cfg or NetworkConfig()
        self._build(); self._init_weights(); self._prewire()

    def _build(self):
        c = self.cfg
        self.W_cat_hid   = nn.Parameter(torch.zeros(c.n_hidden, c.n_category))
        self.W_hid_cat   = nn.Parameter(torch.zeros(c.n_category, c.n_hidden))
        self.W_item_hid  = nn.Parameter(torch.zeros(c.n_hidden, c.n_item))
        self.W_hid_item  = nn.Parameter(torch.zeros(c.n_item, c.n_hidden))
        self.W_hid_out   = nn.Parameter(torch.zeros(c.n_output, c.n_hidden))
        self.W_out_hid   = nn.Parameter(torch.zeros(c.n_hidden, c.n_output))
        self.W_hid_hid   = nn.Parameter(torch.zeros(c.n_hidden, c.n_hidden))
        self.hid_kwta = KWTALayer(c.hidden_k, c.hidden_k_max, c.hidden_target_diff,
                                   c.hidden_osc_amp, c.hidden_osc_period)
        self.out_kwta = KWTALayer(c.output_k, c.output_k_max, c.output_target_diff,
                                   c.output_osc_amp)
        self.lr_rule = BCMLearningRule(c.lr, c.tau_bcm) if c.learning_rule == "bcm"                        else NMPHLearningRule()
        self._last = {}
        self._pairmate_hidden = {}

    def _init_weights(self):
        for p in [self.W_cat_hid, self.W_hid_cat, self.W_item_hid,
                  self.W_hid_item, self.W_hid_hid]:
            nn.init.uniform_(p, 0.45, 0.55)
        for p in [self.W_hid_out, self.W_out_hid]:
            nn.init.uniform_(p, 0.01, 0.03)
        with torch.no_grad():
            self.W_hid_hid.data.fill_diagonal_(0.0)

    def _prewire(self):
        c = self.cfg
        n_h, ov, s = c.n_hidden, c.hidden_overlap, c.prewire_strength
        n_u = (n_h - ov) // 2
        Au = list(range(n_u)); Sh = list(range(n_u, n_u+ov)); Bu = list(range(n_u+ov, 2*n_u+ov))
        with torch.no_grad():
            for u in Au + Sh:
                self.W_item_hid.data[u, 0] = s
            for u in Bu + Sh:
                self.W_item_hid.data[u, 1] = s
            for u in Au:
                for v in Au:
                    if u != v: self.W_hid_hid.data[u, v] = s * 0.5
            for u in Bu:
                for v in Bu:
                    if u != v: self.W_hid_hid.data[u, v] = s * 0.5

    def tick(self, cat, item, hid, out, t=0):
        exc_hid = (self.W_cat_hid @ cat + self.W_item_hid @ item
                   + self.W_hid_hid @ hid + self.W_out_hid @ out)
        exc_out = self.W_hid_out @ hid
        return self.hid_kwta(exc_hid, t), self.out_kwta(exc_out, t)

    def present(self, cat_idx, item_idx, n_ticks=20, label=None):
        c = self.cfg
        cat  = torch.zeros(c.n_category); cat[cat_idx] = 1.0
        item = torch.zeros(c.n_item);     item[item_idx] = 1.0
        hid  = torch.zeros(c.n_hidden)
        out  = torch.zeros(c.n_output)
        for t in range(n_ticks):
            hid, out = self.tick(cat, item, hid, out, t)
        self._last = dict(hid=hid.detach(), out=out.detach(),
                          item=item.detach(), cat=cat.detach())
        if label is not None:
            self._pairmate_hidden[label] = hid.detach().clone()
        return hid, out

    def update_weights(self):
        h, o, it, cat = (self._last[k] for k in ('hid','out','item','cat'))
        with torch.no_grad():
            self.W_item_hid.data  = self.lr_rule(self.W_item_hid.data,  it, h)
            self.W_hid_item.data  = self.lr_rule(self.W_hid_item.data,  h,  it)
            self.W_cat_hid.data   = self.lr_rule(self.W_cat_hid.data,   cat, h)
            self.W_hid_cat.data   = self.lr_rule(self.W_hid_cat.data,   h,  cat)
            self.W_hid_hid.data   = self.lr_rule(self.W_hid_hid.data,   h,  h)
            self.W_hid_out.data   = self.lr_rule(self.W_hid_out.data,   h,  o)
            self.W_out_hid.data   = self.lr_rule(self.W_out_hid.data,   o,  h)
            self.W_hid_hid.data.fill_diagonal_(0.0)

    def similarity(self):
        if 'A' not in self._pairmate_hidden or 'B' not in self._pairmate_hidden:
            return None
        return F.cosine_similarity(
            self._pairmate_hidden['A'].unsqueeze(0),
            self._pairmate_hidden['B'].unsqueeze(0)).item()


## 2. Oscillation amplitude sweep

Sweep `osc_amp` and record pairmate similarity trajectory. Expect:
- **low amp**: little competitor reactivation → no change
- **moderate amp**: moderate reactivation → **differentiation** (similarity decreases)
- **high amp**: strong reactivation → **integration** (similarity increases)

### What to expect

The sweep tests the **nonmonotonic plasticity hypothesis (NMPH)** mechanically.
Oscillation amplitude (`osc_amp`) controls how strongly an inactive competitor memory reactivates
when a target is retrieved. The prediction, following the BCM U-shaped learning rule, is:

| `osc_amp` range | Competitor activity | Predicted outcome |
|---|---|---|
| Very low | Near zero | No representational change |
| Moderate | Moderate | **Differentiation** — Δsim < 0 |
| High | High | **Integration** — Δsim > 0 |

This should produce a **U-shaped curve** when plotting Δsim against `osc_amp`.
The crossover from differentiation to integration is the key boundary to identify —
it will guide the amplitude choices in the naturalistic pipeline (notebook 02).


In [ ]:
N_TRIALS     = 8      # competition trials per amplitude condition
N_TICKS      = 20     # settling steps per trial
N_REPEATS    = 30     # independent model initialisations (stochastic variability)
OMP_AMPS     = np.linspace(0.0, 0.20, 21)  # 0.0 → 0.20 in steps of 0.01

results = {}  # osc_amp → list of (sim_before, sim_after) across repeats

for amp in OMP_AMPS:
    sims_before, sims_after = [], []
    for seed in range(N_REPEATS):
        torch.manual_seed(seed)
        cfg = NetworkConfig(hidden_osc_amp=amp, output_osc_amp=amp * 1.05)
        net = NMPHNetwork(cfg)

        net.present(0, 0, N_TICKS, label='A')
        net.present(0, 1, N_TICKS, label='B')
        sim_before = net.similarity()

        # Competition trials
        for _ in range(N_TRIALS):
            net.present(0, 0, N_TICKS, label='A'); net.update_weights()
            net.present(0, 1, N_TICKS, label='B'); net.update_weights()

        sim_after = net.similarity()
        sims_before.append(sim_before)
        sims_after.append(sim_after)

    results[amp] = dict(before=np.array(sims_before), after=np.array(sims_after))
    delta = np.mean(sims_after) - np.mean(sims_before)
    tag = "DIFF" if delta < -0.02 else "INTG" if delta > 0.02 else "~0  "
    print(f"osc_amp={amp:.3f}  Δsim={delta:+.4f}  [{tag}]")


In [ ]:
# ── Auto-print: sweep summary ──────────────────────────────────────────────
amps_sorted = sorted(results.keys())
deltas_mean = {a: float(results[a]['after'].mean() - results[a]['before'].mean())
               for a in amps_sorted}

diff_amps = [a for a, d in deltas_mean.items() if d < -0.01]
intg_amps = [a for a, d in deltas_mean.items() if d >  0.01]
nc_amps   = [a for a, d in deltas_mean.items() if abs(d) <= 0.01]

max_diff_amp = min(diff_amps, key=lambda a: deltas_mean[a]) if diff_amps else None
crossover    = min(intg_amps) if intg_amps else None
baseline_sim = float(results[amps_sorted[0]]['before'].mean())

print("=" * 60)
print("OSCILLATION AMPLITUDE SWEEP — RESULTS SUMMARY")
print("=" * 60)
print(f"  Baseline pairmate similarity (before any learning): {baseline_sim:+.4f}")
print(f"  No-change zone   : osc_amp ∈ {[round(a,3) for a in nc_amps]}")
print(f"  Differentiation  : osc_amp ∈ {[round(a,3) for a in diff_amps]}")
print(f"  Integration      : osc_amp ∈ {[round(a,3) for a in intg_amps]}")
print()
if diff_amps:
    print(f"  Peak differentiation at osc_amp = {max_diff_amp:.3f}  "
          f"(Δsim = {deltas_mean[max_diff_amp]:+.4f})")
if crossover:
    print(f"  Diff→Intg crossover at osc_amp ≈ {crossover:.3f}")
print()

# Interpret
if diff_amps and intg_amps:
    print("✓  U-shaped NMPH outcome reproduced: differentiation at moderate amplitude,")
    print("   integration at high amplitude — consistent with Ritvo et al. (2024).")
elif diff_amps and not intg_amps:
    print("⚠  Only differentiation observed — try extending osc_amp range beyond 0.20.")
elif intg_amps and not diff_amps:
    print("⚠  Only integration observed — kWTA inhibition may be too weak; try lower lr.")
else:
    print("⚠  No clear directional change — check learning rate (lr) and N_TRIALS.")

print()
print(f"  Recommended DIFF_AMP for trajectory plot: "
      f"{max_diff_amp if max_diff_amp else 0.10:.3f}")
print("=" * 60)


### Plot: similarity change as function of oscillation amplitude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

amps    = sorted(results.keys())
deltas  = [results[a]['after'].mean() - results[a]['before'].mean() for a in amps]
deltas_se = [results[a]['after'].std() / np.sqrt(N_REPEATS) for a in amps]
afters  = [results[a]['after'].mean() for a in amps]
afters_se = [results[a]['after'].std() / np.sqrt(N_REPEATS) for a in amps]

# Left: delta similarity (differentiation = negative)
ax = axes[0]
ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
ax.fill_between(amps,
                np.array(deltas) - np.array(deltas_se),
                np.array(deltas) + np.array(deltas_se),
                alpha=0.25, color='steelblue')
ax.plot(amps, deltas, 'o-', color='steelblue', lw=2, ms=5)
ax.set_xlabel("Oscillation amplitude (osc_amp)", fontsize=12)
ax.set_ylabel("Δ cosine similarity (after − before)", fontsize=12)
ax.set_title("NMPH outcome: differentiation vs integration", fontsize=12)

# Shade zones
diff_zone = [a for a, d in zip(amps, deltas) if d < -0.01]
intg_zone = [a for a, d in zip(amps, deltas) if d > 0.01]
if diff_zone:
    ax.axvspan(min(diff_zone)-0.005, max(diff_zone)+0.005,
               alpha=0.08, color='red', label='Differentiation zone')
if intg_zone:
    ax.axvspan(min(intg_zone)-0.005, max(intg_zone)+0.005,
               alpha=0.08, color='green', label='Integration zone')
ax.legend(fontsize=10)

# Right: absolute similarity after learning
ax = axes[1]
ax.fill_between(amps,
                np.array(afters) - np.array(afters_se),
                np.array(afters) + np.array(afters_se),
                alpha=0.25, color='darkorange')
ax.plot(amps, afters, 's-', color='darkorange', lw=2, ms=5, label='Post-learning')
ax.axhline(results[amps[0]]['before'].mean(), color='grey', ls='--', lw=1.2, label='Pre-learning baseline')
ax.set_xlabel("Oscillation amplitude (osc_amp)", fontsize=12)
ax.set_ylabel("Cosine similarity (A vs B)", fontsize=12)
ax.set_title("Absolute pairmate similarity after competition", fontsize=12)
ax.legend(fontsize=10)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/nmph_osc_amp_sweep.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to /tmp/nmph_osc_amp_sweep.png")


## 3. Single-amplitude trajectory plot

Pick a differentiation-inducing amplitude and trace similarity trial-by-trial.

In [ ]:
# Choose a moderate amplitude (from the sweep above, pick one in the diff zone)
DIFF_AMP = 0.10   # adjust if needed based on sweep results

N_TRAJ_TRIALS = 12
N_SEEDS = 20
trajectories = []

for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    cfg = NetworkConfig(hidden_osc_amp=DIFF_AMP, output_osc_amp=DIFF_AMP * 1.05)
    net = NMPHNetwork(cfg)

    net.present(0, 0, N_TICKS, label='A')
    net.present(0, 1, N_TICKS, label='B')
    traj = [net.similarity()]

    for _ in range(N_TRAJ_TRIALS):
        net.present(0, 0, N_TICKS, label='A'); net.update_weights()
        net.present(0, 1, N_TICKS, label='B'); net.update_weights()
        traj.append(net.similarity())

    trajectories.append(traj)

traj_arr = np.array(trajectories)  # (N_SEEDS, N_TRAJ_TRIALS+1)
traj_mean = traj_arr.mean(0)
traj_se   = traj_arr.std(0) / np.sqrt(N_SEEDS)
traj_min  = traj_arr.min(0)
traj_max  = traj_arr.max(0)

fig, ax = plt.subplots(figsize=(8, 4))
xs = np.arange(len(traj_mean))
ax.fill_between(xs, traj_min, traj_max, alpha=0.15, color='steelblue', label='Range')
ax.fill_between(xs, traj_mean - traj_se, traj_mean + traj_se,
                alpha=0.35, color='steelblue', label='Mean ± SE')
ax.plot(xs, traj_mean, 'o-', color='steelblue', lw=2, ms=6, label='Mean')
ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4, label='Anticorrelation threshold')
ax.set_xlabel("Competition trial", fontsize=12)
ax.set_ylabel("Cosine similarity (A vs B)", fontsize=12)
ax.set_title(f"Pairmate similarity trajectory (osc_amp={DIFF_AMP})", fontsize=12)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Annotate direction
final = traj_mean[-1]
direction = "DIFFERENTIATION" if final < traj_mean[0] - 0.02 else             "INTEGRATION" if final > traj_mean[0] + 0.02 else "NO CHANGE"
ax.text(0.98, 0.05, direction, transform=ax.transAxes, ha='right', fontsize=11,
        color='firebrick' if direction=="DIFFERENTIATION" else 'seagreen')

plt.tight_layout()
plt.savefig("/tmp/nmph_trajectory.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Auto-print: trajectory summary ─────────────────────────────────────────
final_sim   = float(traj_mean[-1])
initial_sim = float(traj_mean[0])
delta_total = final_sim - initial_sim
any_anticorr = float(traj_arr.min()) < 0.0
frac_anticorr = float((traj_arr[:,-1] < 0).mean())

print("=" * 60)
print("TRAJECTORY SUMMARY")
print("=" * 60)
print(f"  osc_amp             : {DIFF_AMP}")
print(f"  Initial pairmate sim: {initial_sim:+.4f}")
print(f"  Final pairmate sim  : {final_sim:+.4f}   (after {N_TRAJ_TRIALS} competition trials)")
print(f"  Total Δsim          : {delta_total:+.4f}")
print(f"  Min sim reached     : {float(traj_arr.min()):+.4f}")
print(f"  Anticorrelation     : {'YES' if any_anticorr else 'not reached'}")
if any_anticorr:
    print(f"  Fraction of seeds reaching anticorrelation: {frac_anticorr*100:.0f}%")
print()

if delta_total < -0.02:
    print("✓  Differentiation confirmed at this amplitude.")
    if any_anticorr:
        print("✓  Anticorrelation reached in at least one seed — matches Ritvo et al.")
        print("   prediction that differentiation manifests as anticorrelated representations.")
    else:
        print("ℹ  Anticorrelation not yet reached — may require more competition trials")
        print("   or a slightly higher amplitude. This is within expected parameter space.")
elif delta_total > 0.02:
    print("ℹ  Integration at this amplitude — try a lower osc_amp for differentiation.")
else:
    print("ℹ  Minimal change at this amplitude — near the no-change zone.")
print("=" * 60)


**Interpreting the trajectory:** Each step on the x-axis is one competition trial (one presentation
of pairmate B while pairmate A can reactivate as a competitor). The y-axis is cosine similarity
between A and B's hidden representations.

- A **downward trajectory** is differentiation — shared connections are weakened because A reactivates
  at moderate activity when B is presented
- Reaching **y < 0** (anticorrelation) is the strongest NMPH prediction: the representations have
  not merely drifted apart but become opposed, because the weakened shared units have been replaced
  by units unique to each pairmate
- The **shaded range** (min–max across seeds) shows stochasticity due to random network initialisation;
  the width of this range indicates how sensitive the outcome is to initial conditions


## 4. Hidden representation visualisation

Inspect which hidden units are assigned to A, B, and shared after competition. Differentiation should show A- and B-specific units becoming more exclusive (fewer shared active units).

In [ ]:
# Pick one seed for visualisation
torch.manual_seed(5)
cfg_diff  = NetworkConfig(hidden_osc_amp=DIFF_AMP,  output_osc_amp=DIFF_AMP * 1.05)
cfg_intg  = NetworkConfig(hidden_osc_amp=0.17, output_osc_amp=0.17 * 1.05)  # adjust to integration zone

def run_and_collect(cfg, n_trials=6):
    net = NMPHNetwork(cfg)
    net.present(0, 0, N_TICKS, label='A')
    net.present(0, 1, N_TICKS, label='B')
    h_A_before = net._pairmate_hidden['A'].numpy().copy()
    h_B_before = net._pairmate_hidden['B'].numpy().copy()
    for _ in range(n_trials):
        net.present(0, 0, N_TICKS, label='A'); net.update_weights()
        net.present(0, 1, N_TICKS, label='B'); net.update_weights()
    h_A_after = net._pairmate_hidden['A'].numpy().copy()
    h_B_after = net._pairmate_hidden['B'].numpy().copy()
    return h_A_before, h_B_before, h_A_after, h_B_after

results_diff = run_and_collect(cfg_diff)
results_intg = run_and_collect(cfg_intg)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
labels_col = ['A (before)', 'B (before)', 'A (after)', 'B (after)']
for row, (results, condition) in enumerate([(results_diff, f'Differentiation (amp={DIFF_AMP})'),
                                             (results_intg, 'Integration (amp=0.17)')]):
    for col, (vec, lbl) in enumerate(zip(results, labels_col)):
        ax = axes[row, col]
        ax.bar(range(len(vec)), vec, color='steelblue' if 'A' in lbl else 'darkorange', alpha=0.7)
        ax.set_title(lbl, fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_xlabel("Hidden unit", fontsize=8)
        if col == 0:
            ax.set_ylabel(condition, fontsize=9, rotation=90, labelpad=5)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle("Hidden layer activations before and after competition", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("/tmp/nmph_hidden_reps.png", dpi=150, bbox_inches='tight')
plt.show()


## 5. BCM vs NMPH piecewise rule comparison

In [ ]:
rules = {'BCM': 'bcm', 'NMPH piecewise': 'nmph'}
AMP_COMPARE = DIFF_AMP
N_TRIALS_COMPARE = 10

rule_results = {}
for rule_name, rule_id in rules.items():
    sims_after = []
    for seed in range(N_REPEATS):
        torch.manual_seed(seed)
        cfg = NetworkConfig(hidden_osc_amp=AMP_COMPARE, output_osc_amp=AMP_COMPARE*1.05,
                            learning_rule=rule_id)
        net = NMPHNetwork(cfg)
        net.present(0, 0, N_TICKS, label='A')
        net.present(0, 1, N_TICKS, label='B')
        for _ in range(N_TRIALS_COMPARE):
            net.present(0, 0, N_TICKS, label='A'); net.update_weights()
            net.present(0, 1, N_TICKS, label='B'); net.update_weights()
        sims_after.append(net.similarity())
    rule_results[rule_name] = np.array(sims_after)
    print(f"{rule_name}: mean sim = {np.mean(sims_after):.4f} ± {np.std(sims_after):.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
for i, (rule_name, sims) in enumerate(rule_results.items()):
    ax.violinplot(sims, positions=[i], showmedians=True)
ax.set_xticks(range(len(rules))); ax.set_xticklabels(list(rules.keys()))
ax.set_ylabel("Post-learning pairmate cosine similarity")
ax.set_title(f"BCM vs NMPH rule (osc_amp={AMP_COMPARE})")
ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/nmph_rule_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Auto-print: learning rule comparison ────────────────────────────────────
print("=" * 60)
print("LEARNING RULE COMPARISON")
print("=" * 60)
for rule_name, sims in rule_results.items():
    direction = ("differentiation" if sims.mean() < baseline_sim - 0.02
                 else "integration" if sims.mean() > baseline_sim + 0.02
                 else "no clear change")
    print(f"  {rule_name:20s}  mean sim = {sims.mean():+.4f} ± {sims.std():.4f}  → {direction}")
print()

bcm_mean  = rule_results['BCM'].mean()
nmph_mean = rule_results['NMPH piecewise'].mean()
diff_rules = abs(bcm_mean - nmph_mean)

if diff_rules < 0.03:
    print("✓  BCM and NMPH piecewise rules produce qualitatively similar outcomes,")
    print("   supporting the equivalence of these two U-shaped learning formulations.")
    print("   BCM is used in the naturalistic pipeline (simpler; sliding threshold).")
else:
    print(f"ℹ  Rules diverge by Δsim = {diff_rules:.4f}.")
    more_diff = 'BCM' if bcm_mean < nmph_mean else 'NMPH piecewise'
    print(f"   {more_diff} produces stronger differentiation at this amplitude.")
    print("   Consider which rule to use in the naturalistic pipeline (see docs/decisions.md).")
print("=" * 60)


**Interpreting the comparison:** Both rules implement the same core principle — a U-shaped
relationship between synaptic activity and weight change — but differ in their formulation:

- **BCM** uses a sliding modification threshold θ_M (exponential moving average of y²).
  The threshold adapts over time, making the rule self-regulating: in a high-activity regime
  the threshold rises, pulling more units into the integration zone.
- **NMPH piecewise** uses fixed thresholds (θ_low, θ_cross), giving more direct control but
  requiring manual tuning per dataset.

If the two rules produce similar outcomes here, BCM is the safer default for the naturalistic
pipeline — it is more robust to the varied activity levels that emerge from real video features.
If they diverge substantially, inspect which amplitude regime drives the difference and decide
whether the sliding threshold is appropriate for your data.


## 6. Push to GitHub

In [ ]:
# =============================================================================
#  GitHub Sync — Push  (run whenever you want to save to GitHub)
# =============================================================================
import json as _json, os as _os, subprocess as _sp
from datetime import datetime as _dt
from google.colab import _message

_nb   = _message.blocking_request('get_ipynb', timeout_sec=30)
_dest = f"{REPO_PATH}/{NOTEBOOK_REL}"
_os.makedirs(_os.path.dirname(_dest), exist_ok=True)
with open(_dest, 'w') as _f:
    _json.dump(_nb, _f, indent=1)

_msg = f"[colab] 01_nmph_toy_validation: {_dt.now():%Y-%m-%d %H:%M}"
_sp.run(["git", "-C", REPO_PATH, "add", NOTEBOOK_REL], check=True)
_res = _sp.run(["git", "-C", REPO_PATH, "commit", "-m", _msg],
               capture_output=True, text=True)
if "nothing to commit" in _res.stdout:
    print("Nothing to commit — notebook unchanged.")
else:
    _sp.run(["git", "-C", REPO_PATH, "push"], check=True)
    print(f"Pushed: {_msg}")
